In [6]:
import csv, json, os, pickle, time, numpy as np
from sentence_transformers import SentenceTransformer
from annoy import AnnoyIndex
from spellchecker import SpellChecker

# Load Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

In [16]:
# Preload the spellchecker with domain-specific words
spell = SpellChecker()
spell.word_frequency.load_words(["oil", "massage", "hot", "stone"])
# Custom corrections mapping
CUSTOM_CORRECTIONS = {
    #"oli": "oil",
    #"message": "massage",
    #"mesaage": "massage",
    #"mesage": "massage"
}

def load_data(filename, delimiter=','):
    if filename.endswith('.csv'):
        with open(filename, newline='', encoding='utf-8') as f:
            return list(csv.DictReader(f, delimiter=delimiter))
    elif filename.endswith('.json'):
        with open(filename, 'r', encoding='utf-8') as f:
            return json.load(f)
    else:
        raise ValueError("Unsupported file format")

def precompute_embeddings(data, field='description'):
    texts = [entry.get(field, "") for entry in data]
    return model.encode(texts, show_progress_bar=True)

def save_pickle(obj, filepath):
    with open(filepath, 'wb') as f:
        pickle.dump(obj, f)

def load_pickle(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

def correct_query(query):
    corrected_words = []
    for word in query.split():
        lw = word.lower()
        if lw in CUSTOM_CORRECTIONS:
            corrected_words.append(CUSTOM_CORRECTIONS[lw])
        else:
            corrected_words.append(spell.correction(word))
    return " ".join(corrected_words)

def build_annoy_index(vectors, num_trees=10, index_path="annoy_index.ann"):
    dim = vectors.shape[1]
    t = AnnoyIndex(dim, 'angular')
    for i, vector in enumerate(vectors):
        t.add_item(i, vector)
    t.build(num_trees)
    t.save(index_path)
    return t

def load_annoy_index(dim, index_path="annoy_index.ann"):
    t = AnnoyIndex(dim, 'angular')
    t.load(index_path)
    return t

# Main block
filename = "data.csv"  # Your file with a 'description' column
data = load_data(filename)

# Load or compute embeddings
emb_file = "embeddings_st.pkl"
if os.path.exists(emb_file):
    desc_vectors = load_pickle(emb_file)
else:
    desc_vectors = precompute_embeddings(data, field='description')
    save_pickle(desc_vectors, emb_file)

dim = desc_vectors.shape[1]
annoy_index_path = "annoy_index.ann"
if os.path.exists(annoy_index_path):
    t = load_annoy_index(dim, index_path=annoy_index_path)
else:
    t = build_annoy_index(desc_vectors, num_trees=10, index_path=annoy_index_path)

In [19]:
# Process query
query = "oli chage"
start_time = time.time()
corrected_query = correct_query(query)
print("Corrected query:", corrected_query)
query_vec = model.encode([corrected_query])[0]

# Retrieve top 3 candidates using Annoy
results = t.get_nns_by_vector(query_vec, 3, include_distances=True)
top_indices = results[0]
top_distances = results[1]

for idx, d in zip(top_indices, top_distances):
    # Approximate cosine similarity from angular distance
    cos_sim = 1 - (d**2 / 2)
    print("Description:", data[idx]['description'])
    print("Similarity:", cos_sim)
    print("-----")

end_time = time.time()
print("Query processed in {:.2f} ms".format((end_time - start_time) * 1000))

Corrected query: old change
Description: Oil Change
Similarity: 0.5415088277920841
-----
Description: Alternative
Similarity: 0.3996838922612369
-----
Description: Pension
Similarity: 0.3104200725277835
-----
Query processed in 893.00 ms
